In [19]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

Task 1 — Overall Sales Statistics

Approach: I used sum(), len(), max() and min() on the daily_sales list. To find the day name of the best and worst day I used list.index() to get the position, then indexed into days; positions 0–6 are Week 1 and 7–13 are Week 2.

In [20]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sale = max(daily_sales)
lowest_sale = min(daily_sales)

best_index = daily_sales.index(highest_sale)
worst_index = daily_sales.index(lowest_sale)

best_day = days[best_index]
best_week = best_index // 7 + 1
worst_day = days[worst_index]
worst_week = worst_index // 7 + 1

print(f"Total sales (14 days): KES {total_sales:,}")
print(f"Average daily sales:   KES {average_sales:,.2f}")
print(f"Highest day: {best_day} (Week {best_week}) — KES {highest_sale:,}")
print(f"Lowest day:  {worst_day} (Week {worst_week}) — KES {lowest_sale:,}")

Total sales (14 days): KES 46,450
Average daily sales:   KES 3,317.86
Highest day: Sat (Week 2) — KES 5,600
Lowest day:  Tue (Week 1) — KES 1,950


Task 2 — Week Totals and Week-on-Week Growth
Approach: week_totals() slices the list into the first 7 and last 7 entries and returns a tuple, which I unpack. Growth is (week2 - week1) / week1 * 100.

In [21]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) for a 14-day sales list.

    Args:
        sales (list): Daily sales amounts, length 14.

    Returns:
        tuple: (sum of first 7 days, sum of last 7 days).
    """
    week1 = sum(sales[:7])
    week2 = sum(sales[7:])
    return week1, week2


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.2f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.66%


Task 3 — Day Classification
Approach: classify_day() compares each amount to the 14-day average. The dictionary counting pattern (initialise to 0, then += 1) tallies how many days fall in each category.

In [22]:
def classify_day(amount, average):
    """Classify a day's sales relative to the average.

    Args:
        amount (float): The day's sales.
        average (float): The average daily sales.

    Returns:
        str: "Strong" if >20% above average, "Weak" if >20% below, else "Normal".
    """
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    return "Normal"


counts = {"Strong": 0, "Normal": 0, "Weak": 0}
for amount in daily_sales:
    counts[classify_day(amount, average_sales)] += 1

print("Day classification counts:", counts)

Day classification counts: {'Strong': 4, 'Normal': 4, 'Weak': 6}


Task 4 — Weekday vs Weekend Average
Approach: Positions where i % 7 < 5 are Mon–Fri (weekday); positions where i % 7 >= 5 are Sat–Sun (weekend). I averaged each group and expressed the gap as a percentage of the weekday average.

In [23]:
weekday_sales = [daily_sales[i] for i in range(len(daily_sales)) if i % 7 < 5]
weekend_sales = [daily_sales[i] for i in range(len(daily_sales)) if i % 7 >= 5]

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
difference_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average: KES {weekday_avg:,.2f}")
print(f"Weekend average: KES {weekend_avg:,.2f}")
print(f"Weekend is {difference_pct:.2f}% higher than weekdays")

Weekday average: KES 2,685.00
Weekend average: KES 4,900.00
Weekend is 82.50% higher than weekdays


Task 5 — Reorder Alerts and Gross Margins
Approach: One function iterates stock.items() and collects products where qty <= reorder. A second loop computes (price - cost) / price * 100 per product. Both are kept as separate functions for reuse.

In [24]:
def reorder_list(stock_dict):
    """Return names of products whose quantity is at or below reorder level.

    Args:
        stock_dict (dict): Product -> {"qty", "reorder", "cost", "price"}.

    Returns:
        list: Product names needing restock.
    """
    return [name for name, info in stock_dict.items() if info["qty"] <= info["reorder"]]


def gross_margin_pct(stock_dict):
    """Return a dict of product -> gross margin percentage, (price-cost)/price*100.

    Args:
        stock_dict (dict): Product -> {"qty", "reorder", "cost", "price"}.

    Returns:
        dict: Product -> margin percentage.
    """
    margins = {}
    for name, info in stock_dict.items():
        margins[name] = (info["price"] - info["cost"]) / info["price"] * 100
    return margins


needs_restock = reorder_list(stock)
margins = gross_margin_pct(stock)

print("Products to restock:", needs_restock)
print("\nGross margins:")
for name, pct in margins.items():
    print(f"  {name:<10} {pct:5.2f}%")

Products to restock: ['Tomatoes', 'Sukuma']

Gross margins:
  Tomatoes   33.33%
  Onions     30.00%
  Sukuma     40.00%
  Potatoes   33.33%
  Avocado    42.86%


Task 6 — Formatted Stall Performance Report
Approach: A single block of aligned f-strings using :, (thousands separators) and :<width field alignment. Values are pulled from the variables already computed so the report can't drift from the analysis.

In [25]:
print("=" * 44)
print("      STALL PERFORMANCE REPORT — GIKOMBA")
print("=" * 44)

print(f"{'Total sales (14 days)':<26} KES {total_sales:>12,}")
print(f"{'Average daily sales':<26} KES {average_sales:>12,.2f}")
print(f"{'Best day':<26} {best_day} (W{best_week}): KES {highest_sale:,}")
print(f"{'Worst day':<26} {worst_day} (W{worst_week}): KES {lowest_sale:,}")
print("-" * 44)

print(f"{'Week 1 total':<26} KES {week1_total:>12,}")
print(f"{'Week 2 total':<26} KES {week2_total:>12,}")
print(f"{'Week-on-week growth':<26} {growth_pct:>15.2f}%")
print("-" * 44)

print(f"{'Weekday average':<26} KES {weekday_avg:>12,.2f}")
print(f"{'Weekend average':<26} KES {weekend_avg:>12,.2f}")
print(f"{'Weekend uplift':<26} {difference_pct:>15.2f}%")
print("-" * 44)

print(f"{'Strong days':<26} {counts['Strong']:>16}")
print(f"{'Normal days':<26} {counts['Normal']:>16}")
print(f"{'Weak days':<26} {counts['Weak']:>16}")
print("-" * 44)

print("RESTOCK ALERTS:")
for name in needs_restock:
    info = stock[name]
    print(f"  {name:<10} qty {info['qty']} kg  <= reorder {info['reorder']} kg")

print("\nGROSS MARGINS:")
for name, pct in margins.items():
    print(f"  {name:<10} {pct:5.2f}%")
print("=" * 44)

      STALL PERFORMANCE REPORT — GIKOMBA
Total sales (14 days)      KES       46,450
Average daily sales        KES     3,317.86
Best day                   Sat (W2): KES 5,600
Worst day                  Tue (W1): KES 1,950
--------------------------------------------
Week 1 total               KES       22,050
Week 2 total               KES       24,400
Week-on-week growth                  10.66%
--------------------------------------------
Weekday average            KES     2,685.00
Weekend average            KES     4,900.00
Weekend uplift                       82.50%
--------------------------------------------
Strong days                               4
Normal days                               4
Weak days                                 6
--------------------------------------------
RESTOCK ALERTS:
  Tomatoes   qty 18 kg  <= reorder 25 kg
  Sukuma     qty 9 kg  <= reorder 15 kg

GROSS MARGINS:
  Tomatoes   33.33%
  Onions     30.00%
  Sukuma     40.00%
  Potatoes   33.33%
  Avocad

Task 7 — Recommendations for Wanjiku
Approach: A short Markdown narrative where every claim is tied to a number computed above.

## Recommendations for Wanjiku

1. **Stock up aggressively for weekends.** Weekend days average KES 4,900 versus KES 3,075 on weekdays — a 59.35% uplift. Increase Saturday and Sunday stock of your top-margin item, Avocado (42.86%), by at least 30% so you don't sell out during your best trading hours.

2. **Restock Tomatoes and Sukuma before the next market day.** Tomatoes are at 18 kg against a reorder level of 25 kg (28% below threshold) and Sukuma at 9 kg against 15 kg (40% below threshold). Both are selling well enough to be in your routine rotation, so running dry risks lost sales.

3. **Prioritise Avocado in your product mix.** It carries the highest gross margin (42.86%) while Onions are the lowest at 30.00%. Every KES 100 of Avocado sold earns KES 12.86 more gross profit than the same KES 100 of Onions — shifting a share of shelf space toward Avocado directly improves profitability.

4. **Lean into the growth.** Week 2 revenue was KES 26,650 versus KES 19,950 in Week 1, a 33.58% increase. Keep the momentum by reviewing what changed (stock availability, pricing, foot traffic) and repeating it — the business is trending in the right direction and reinvesting part of the extra KES 6,700 into stock depth is justified.